# 01 Preprocessing — Stage 1 Dataset Build

Runs the v5 individual-level preprocessing pipeline (724,115 rows) and exports `data/processed/X_features.csv` plus `y_*.csv` targets.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_stage1_data
from src.preprocessing.clean import handle_missing
from src.preprocessing.target_builder import build_targets
from src.preprocessing.engineer_features import engineer_features
from src.preprocessing.encode import encode_features

print('Project root:', PROJECT_ROOT)

Project root: b:\PBCS\Major Project\BarrierLens_MP_G25_P48


In [2]:
df = load_stage1_data()
print('Loaded shape:', df.shape)
df.head(2)

b:\PBCS\Major Project\BarrierLens_MP_G25_P48\src\preprocessing\load_data.py:96: UserWarning: Columns not found in CSV and skipped: ['v467f', 'v467i', 'm14']
  warnings.warn(f"Columns not found in CSV and skipped: {missing}")


Loaded CSV file: NFHS5_Individual.csv -> (724115, 32)
Analytic sample: full file (724,115 rows)
Loaded shape: (724115, 32)


,caseid,v001,v002,v021,v024,v025,v012,v013,v106,v130,...,v467b,v467c,v467d,v467e,v467g,v467h,v626a,s245a,s245b,s245h
0,0100101305 04,113,5,113,jammu & kashmir,rural,22,20-24,higher,muslim,...,big problem,not a big problem,not a big problem,big problem,big problem,big problem,never had sex,NaN,NaN,NaN
1,0100101305 05,113,5,113,jammu & kashmir,rural,19,15-19,secondary,muslim,...,big problem,big problem,big problem,big problem,big problem,big problem,never had sex,NaN,NaN,NaN


In [3]:
df = handle_missing(df)
df = build_targets(df)
df = engineer_features(df)
encoded = encode_features(df)

print('Post-encoding feature matrix shape:', encoded.shape)
print('Target prevalence:')
for t in ['target_household', 'target_logistic', 'target_facility']:
    print(t, df[t].mean().round(4), df[t].value_counts().to_dict())

Dropped v466 (100% missing in this extract)
v131: filled 4373 missing values with 'missing'
v717: filled 615461 missing values with 'missing'
v169a: filled 615330 missing values with 'missing'
v170: filled 615330 missing values with 'missing'
v743f: filled 648289 missing values with 'missing'
Feature missing values remaining: 0


b:\PBCS\Major Project\BarrierLens_MP_G25_P48\src\preprocessing\target_builder.py:28: UserWarning: target_household: missing barrier columns ['v467f']; building from available columns only.
  warnings.warn(


target_household: positive rate = 0.2716 (27.2%) | built from ['v467b', 'v467c'] | counts = {0: 527477, 1: 196638}
target_logistic: positive rate = 0.3161 (31.6%) | built from ['v467d', 'v467e'] | counts = {0: 495248, 1: 228867}


b:\PBCS\Major Project\BarrierLens_MP_G25_P48\src\preprocessing\target_builder.py:28: UserWarning: target_facility: missing barrier columns ['v467i']; building from available columns only.
  warnings.warn(


target_facility: positive rate = 0.4601 (46.0%) | built from ['v467g', 'v467h'] | counts = {0: 390968, 1: 333147}
Post-encoding feature matrix shape: (724115, 37)
Target prevalence:
target_household 0.2716 {0: 527477, 1: 196638}
target_logistic 0.3161 {0: 495248, 1: 228867}
target_facility 0.4601 {0: 390968, 1: 333147}


In [4]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

encoded.to_csv(processed_dir / 'X_features.csv', index=False)
for t in ['target_household', 'target_logistic', 'target_facility']:
    df[t].to_csv(processed_dir / f'y_{t.replace("target_", "")}.csv', index=False)

print('Saved processed files to:', processed_dir)
print('X_features shape:', encoded.shape)
for t in ['target_household', 'target_logistic', 'target_facility']:
    print(t, df[t].value_counts().to_dict())

Saved processed files to: b:\PBCS\Major Project\BarrierLens_MP_G25_P48\data\processed
X_features shape: (724115, 37)
target_household {0: 527477, 1: 196638}
target_logistic {0: 495248, 1: 228867}
target_facility {0: 390968, 1: 333147}
